In [1]:
import torch
import accelerate

print("=== GPU 状态检查 ===")
print(f"CUDA 是否可用: {torch.cuda.is_available()}")
print(f"GPU 数量: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        gpu_name = torch.cuda.get_device_name(i)
        gpu_memory = torch.cuda.get_device_properties(i).total_memory / 1024**3
        print(f"GPU {i}: {gpu_name}")
        print(f"  显存: {gpu_memory:.1f} GB")
    device = torch.device("cuda")
    print(f"✅ 使用GPU进行训练")
else:
    device = torch.device("cpu")
    print("❌ 使用CPU进行训练，速度会较慢")

=== GPU 状态检查 ===
CUDA 是否可用: True
GPU 数量: 1
GPU 0: NVIDIA GeForce RTX 5060 Laptop GPU
  显存: 8.0 GB
✅ 使用GPU进行训练


In [2]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch
import pandas as pd

# 加载训练好的模型
model_path = "my_final_model/my_model_3"
model = AutoModelForSequenceClassification.from_pretrained(model_path)
tokenizer = AutoTokenizer.from_pretrained(model_path)

print("✅ 模型加载成功！")

# 情绪映射和表情符号
emotion_mapping = {
    "开心": 0, "平静": 1, "伤心": 2, "生气": 3, 
    "惊讶": 4, "疑问": 5, "厌恶": 6, "关心": 7,
}

emotion_emojis = {
    "开心": "😊", "平静": "😐", "伤心": "😢", "生气": "😠",
    "惊讶": "😲", "疑问": "🤔", "厌恶": "🤢", "关心": "❤️"
}

def predict_emotion(text):
    """预测单条文本的情绪"""
    # 编码文本
    inputs = tokenizer(
        text, 
        return_tensors="pt", 
        truncation=True, 
        padding=True, 
        max_length=128
    )
    
    # 模型预测
    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)
        predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)
    
    # 获取所有情绪的概率
    probabilities = predictions[0].numpy()
    
    # 找到主要情绪
    predicted_class = probabilities.argmax()
    confidence = probabilities[predicted_class]
    emotion = list(emotion_mapping.keys())[predicted_class]
    emoji = emotion_emojis.get(emotion, "❓")
    
    # 获取所有情绪的概率（按置信度排序）
    emotion_probs = []
    for i, prob in enumerate(probabilities):
        emotion_name = list(emotion_mapping.keys())[i]
        emotion_emoji = emotion_emojis.get(emotion_name, "❓")
        emotion_probs.append({
            'emotion': emotion_name,
            'emoji': emotion_emoji,
            'confidence': prob
        })
    
    emotion_probs.sort(key=lambda x: x['confidence'], reverse=True)
    
    return {
        'text': text,
        'primary_emotion': emotion,
        'primary_emoji': emoji,
        'primary_confidence': confidence,
        'all_emotions': emotion_probs
    }

# 测试各种情绪
test_texts = [
    "太多的事，慢慢地就不能做了，太多的人，渐渐地就不见了。原来，成长就注定是一个要丢失的过程。",
    "你问我为何时常沉默，有的人无话可说，有的话无人可说", 
    "我的目光化作一件无形的衣裳，悄悄披在你肩上，生怕世间的风雨将你吹凉。那是悄无声息的守望，如月光浸润大地。",
    "相信奇迹的人，本身跟奇迹一样了不起",
    "仿佛有什么不洁的东西黏着在感官上，像目睹了玫瑰的糜烂。一种源自心底的疏离，想将这一切从视野里彻底擦去。",
    "我的不耐烦应该已经表现得很明显了",
    "胸腔里仿佛困着一头咆哮的兽，灼热的气息炙烤着理智的弦。怒火是无声的雷鸣，在压抑的乌云里翻滚。",
]

print("🧪 情绪分类测试结果（带表情）:")
print("=" * 70)

for text in test_texts:
    result = predict_emotion(text)
    
    print(f"📝 文本: {result['text']}")
    print(f"🎯 主要情绪: {result['primary_emoji']} {result['primary_emotion']} (置信度: {result['primary_confidence']:.2%})")
    
    print("📊 详细概率:")
    for i, emotion_info in enumerate(result['all_emotions'][:3]):  # 显示前3个
        print(f"   {i+1}. {emotion_info['emoji']} {emotion_info['emotion']}: {emotion_info['confidence']:.2%}")
    
    print("-" * 70)

✅ 模型加载成功！
🧪 情绪分类测试结果（带表情）:
📝 文本: 太多的事，慢慢地就不能做了，太多的人，渐渐地就不见了。原来，成长就注定是一个要丢失的过程。
🎯 主要情绪: 😢 伤心 (置信度: 98.18%)
📊 详细概率:
   1. 😢 伤心: 98.18%
   2. 😐 平静: 0.61%
   3. ❤️ 关心: 0.29%
----------------------------------------------------------------------
📝 文本: 你问我为何时常沉默，有的人无话可说，有的话无人可说
🎯 主要情绪: 😢 伤心 (置信度: 88.96%)
📊 详细概率:
   1. 😢 伤心: 88.96%
   2. 🤔 疑问: 5.76%
   3. 😐 平静: 2.31%
----------------------------------------------------------------------
📝 文本: 我的目光化作一件无形的衣裳，悄悄披在你肩上，生怕世间的风雨将你吹凉。那是悄无声息的守望，如月光浸润大地。
🎯 主要情绪: 😐 平静 (置信度: 58.53%)
📊 详细概率:
   1. 😐 平静: 58.53%
   2. 😢 伤心: 36.69%
   3. ❤️ 关心: 1.60%
----------------------------------------------------------------------
📝 文本: 相信奇迹的人，本身跟奇迹一样了不起
🎯 主要情绪: ❤️ 关心 (置信度: 67.45%)
📊 详细概率:
   1. ❤️ 关心: 67.45%
   2. 😐 平静: 10.75%
   3. 😊 开心: 8.07%
----------------------------------------------------------------------
📝 文本: 仿佛有什么不洁的东西黏着在感官上，像目睹了玫瑰的糜烂。一种源自心底的疏离，想将这一切从视野里彻底擦去。
🎯 主要情绪: 😢 伤心 (置信度: 91.73%)
📊 详细概率:
   1. 😢 伤心: 91.73%
   2. 🤢 厌恶: 3.83%
   3. 😐 平静: 2.23%
--------

In [4]:
def evaluate_with_test_set(test_csv_path):
    """使用测试集评估模型性能"""
    try:
        test_df = pd.read_csv(test_csv_path)
        print(f"📊 使用测试集评估: {len(test_df)} 条数据")
        
        correct = 0
        total = len(test_df)
        
        for index, row in test_df.iterrows():
            text = row['text']
            true_emotion = row['label']
            
            result = predict_emotion(text)
            predicted_emotion = result['primary_emotion']
            
            if predicted_emotion == true_emotion:
                correct += 1
        
        accuracy = correct / total
        print(f"✅ 测试集准确率: {accuracy:.2%} ({correct}/{total})")
        
    except Exception as e:
        print(f"❌ 测试集评估失败: {e}")

# 如果你有测试集，取消注释运行
# evaluate_with_test_set("chinese_emotion/test.csv")

❌ 测试集评估失败: [Errno 2] No such file or directory: 'chinese_emotion/test.csv'


In [ ]:
def interactive_test():
    """交互式情绪测试"""
    print("\n💬 交互式情绪测试")
    print("输入文本分析情绪，输入 '退出' 结束")
    print("=" * 50)
    
    while True:
        text = input("\n请输入文本: ").strip()
        
        if text in ['退出', 'quit', 'exit']:
            print("再见！👋")
            break
            
        if not text:
            continue
            
        result = predict_emotion(text)
        
        print(f"\n📊 分析结果:")
        print(f"   文本: {text}")
        print(f"   主要情绪: {result['primary_emoji']} {result['primary_emotion']} ({result['primary_confidence']:.2%})")
        
        print("   其他可能情绪:")
        for i, emotion_info in enumerate(result['all_emotions'][1:4]):  # 显示第2-4个
            print(f"     {emotion_info['emoji']} {emotion_info['emotion']}: {emotion_info['confidence']:.2%}")

# 取消注释以启用交互式测试
interactive_test()


💬 交互式情绪测试
输入文本分析情绪，输入 '退出' 结束



请输入文本:  我是终将升起的烈阳！



📊 分析结果:
   文本: 我是终将升起的烈阳！
   主要情绪: 😊 开心 (95.59%)
   其他可能情绪:
     😲 惊讶: 1.77%
     😢 伤心: 0.69%
     ❤️ 关心: 0.66%



请输入文本:  我草泥马



📊 分析结果:
   文本: 我草泥马
   主要情绪: 😐 平静 (74.89%)
   其他可能情绪:
     😢 伤心: 5.84%
     😲 惊讶: 5.23%
     🤢 厌恶: 4.29%


...


📊 分析结果:
   文本: 我真想揍你一顿
   主要情绪: 😠 生气 (84.62%)
   其他可能情绪:
     🤢 厌恶: 13.09%
     😐 平静: 0.51%
     😲 惊讶: 0.40%



请输入文本:  我要杀了你！！



📊 分析结果:
   文本: 我要杀了你！！
   主要情绪: 😠 生气 (96.36%)
   其他可能情绪:
     🤢 厌恶: 1.35%
     😊 开心: 0.62%
     😲 惊讶: 0.57%



请输入文本:  十七张牌你能秒杀我？我当场把这个电脑屏幕给吃掉



📊 分析结果:
   文本: 十七张牌你能秒杀我？我当场把这个电脑屏幕给吃掉
   主要情绪: 😠 生气 (54.34%)
   其他可能情绪:
     🤢 厌恶: 34.37%
     😲 惊讶: 6.19%
     😐 平静: 1.99%
